In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn import datasets
from sklearn.metrics import classification_report, confusion_matrix,r2_score,accuracy_score
from sklearn.preprocessing import StandardScaler

In [2]:
df = datasets.load_diabetes(as_frame=True).frame

In [3]:
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [5]:
X = df.drop(columns=["target"])
Y = df["target"]

X_train,X_test,Y_train,Y_test = train_test_split(
    X,Y,
    test_size=0.3,
    random_state=42,
)

In [14]:
Y_scaler = StandardScaler()

Y_train_scaled = Y_scaler.fit_transform(Y_train.values.reshape(-1,1)).ravel()
Y_test_scaled = Y_scaler.transform(Y_test.values.reshape(-1,1)).ravel()

In [15]:
model = SVR()
model.fit(X_train,Y_train_scaled)

SVR()

In [18]:
Y_pred_scaled = model.predict(X_test)

print(r2_score(Y_test_scaled,Y_pred_scaled))

0.48844443151651884


# GridSearchCV for Hyperparameter Tuning

In [19]:
from sklearn.model_selection import GridSearchCV

In [24]:
param_grid = {
    "C":[1,2,3,4,5,6,7,8,9,10],
    "kernel":["rbf","linear","sigmoid","polynoimal"],
    "epsilon":[0.01,0.1,0.2,0.3,0.4,0.5]
}

In [27]:
model = SVR()

grid_search = GridSearchCV(
    model,
    param_grid,
    scoring="r2",
    cv=10
)

grid_search.fit(X_train,Y_train_scaled)

c:\Users\Asus\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
600 fits failed out of a total of 2400.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
600 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Asus\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Asus\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "c:\Users\Asus\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "c:\Users\Asus\anaconda3\Lib\site-packages\s

GridSearchCV(cv=10, estimator=SVR(),
             param_grid={'C': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                         'epsilon': [0.01, 0.1, 0.2, 0.3, 0.4, 0.5],
                         'kernel': ['rbf', 'linear', 'sigmoid', 'polynoimal']},
             scoring='r2')

In [28]:
print("Best Params: ",grid_search.best_params_)

Best Params:  {'C': 5, 'epsilon': 0.2, 'kernel': 'linear'}


In [30]:
best_model = SVR(
    kernel=grid_search.best_params_["kernel"],
    C=grid_search.best_params_["C"],
    epsilon=grid_search.best_params_["epsilon"]
)

best_model.fit(X_train,Y_train_scaled)
Y_test_pred_scaled = best_model.predict(X_test)
Y_train_pred_scaled = best_model.predict(X_train)

print("Train r2:",r2_score(Y_train_scaled,Y_train_pred_scaled))
print("Test r2:",r2_score(Y_test_scaled,Y_test_pred_scaled))

Train r2: 0.5140351527957079
Test r2: 0.48325803166871617
